## Phase 3: Apply Mapping to Both DataFrames

In [2]:
import pandas as pd


In [3]:
usda_clean = pd.read_csv("usda_data_dedup_final.csv") # insert your right usda dataset csv

In [ ]:
# ── Step 8: Apply the mapping to both dataframes ─────────────────────────
# Load mapping (so this cell works even if you restart the kernel)
cat_map = pd.read_csv("category_mapping2.csv")
l2_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l2"]))
l1_lookup = dict(zip(cat_map["original_cat"], cat_map["cat_l1"]))

# Map L1 and L2 onto both dataframes; L3 = original cat
usda_clean["cat_l1"] = usda_clean["cat"].map(l1_lookup)
usda_clean["cat_l2"] = usda_clean["cat"].map(l2_lookup)
usda_clean["cat_l3"] = usda_clean["cat"]

print("\n=== USDA coverage ===")
print(f"  L1 mapped: {usda_clean['cat_l1'].notna().sum():,} / {len(usda_clean):,} ({usda_clean['cat_l1'].notna().mean()*100:.1f}%)")
print(f"  L2 mapped: {usda_clean['cat_l2'].notna().sum():,} / {len(usda_clean):,}")
print(f"  L1 unmapped: {usda_clean['cat_l1'].isna().sum():,} rows")


=== USDA coverage ===
  L1 mapped: 127,728 / 130,054 (98.2%)
  L2 mapped: 127,728 / 130,054
  L1 unmapped: 2,326 rows


In [5]:
# ── Step 9: Handle unmapped rows (tiny categories) ───────────────────────
# For rows where cat wasn't in our mapping (the <3 item categories),
# classify them by item_name using the same LLM function, or assign "other"

unmapped_usda = usda_clean[usda_clean["cat_l2"].isna()]

print(f"Unmapped USDA rows: {len(unmapped_usda)} ({len(unmapped_usda)/len(usda_clean)*100:.1f}%)")

# Option A: assign "other" (simple, fast)
usda_clean["cat_l1"].fillna("other", inplace=True)
usda_clean["cat_l2"].fillna("other", inplace=True)

print("\nAfter filling unmapped → 'other':")
print(f"  USDA L1 unique: {usda_clean['cat_l1'].nunique()}")
print(f"  USDA L2 unique: {usda_clean['cat_l2'].nunique()}")

Unmapped USDA rows: 2326 (1.8%)

After filling unmapped → 'other':
  USDA L1 unique: 20
  USDA L2 unique: 79


/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_30692/4167241119.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  usda_clean["cat_l1"].fillna("other", inplace=True)
/var/folders/yz/grwlw1c90b1dbggwlxd8jf200000gn/T/ipykernel_30692/4167241119.py:11: ChainedAssignmentError: A value is being set on a copy of a DataFrame or

In [6]:
# ── Step 10: Validate — spot check a few L1 groups ───────────────────────


for l1 in ["dairy & eggs", "fruits", "snacks", "prepared & frozen meals", "beverages"]:
    subset = usda_clean[usda_clean["cat_l1"] == l1]
    sample = subset.sample(min(5, len(subset)), random_state=42)
    print(f"\n{'='*60}")
    print(f"L1: {l1}  ({len(subset):,} rows)")
    print(f"{'='*60}")
    print(sample[["item_name", "cat_l1", "cat_l2", "cat_l3", "source"]].to_string(index=False))


L1: dairy & eggs  (14,204 rows)
                        item_name       cat_l1                    cat_l2                                cat_l3 source
        blanc cheese grue gruyere dairy & eggs                    cheese                                cheese   usda
         coconut ice rosati water dairy & eggs ice cream & frozen yogurt             ice cream & frozen yogurt   usda
dairy gingerbread topping whipped dairy & eggs            dairy desserts baking decorations & dessert toppings   usda
           coconut mousse vanilla dairy & eggs            dairy desserts                 other frozen desserts   usda
             bacon cheese pimento dairy & eggs                    cheese                                cheese   usda

L1: fruits  (596 rows)
                     item_name cat_l1                    cat_l2       cat_l3 source
cinnamon del halves monte pear fruits canned & preserved fruits canned fruit   usda
        in longan richin syrup fruits canned & preserved fruits can

In [ ]:
# ── Step 11: Save final datasets ─────────────────────────────────────────
usda_clean.to_csv("usda_dedup_ontology2.csv", index=False)

print(f"Saved usda_dedup_ontology.csv ({len(usda_clean):,} rows)")
print(f"\nColumns: {usda_clean.columns.tolist()}")

Saved usda_dedup_ontology.csv (130,054 rows)

Columns: ['item_name', 'cat', 'carbs_100g', 'kcal_100g', 'fat_100g', 'protein_100g', 'source', 'cat_word_count', 'item_name_word_count', 'cat_l1', 'cat_l2', 'cat_l3']
